#TRAINING THE MODEL

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# LOAD LABELS AND OPTIONAL METADATA
# Load the main labels file
!cp -r "/content/drive/MyDrive/ENGG 680/HAM10000/prepared_images" /content/
!cp "/content/drive/MyDrive/ENGG 680/HAM10000/prepared_images_labels.csv" /content/


In [ ]:
labels_df = pd.read_csv( "/content/prepared_images_labels.csv")  # Change path if needed


In [ ]:
# Change string lesion labels into integers

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels_df['label_int'] = le.fit_transform(labels_df['label'])   # 'label' is the string column
num_classes = labels_df['label_int'].nunique()


In [ ]:
# === 2. DEFINE PYTORCH DATASET CLASS ===
class SkinLesionDataset(Dataset):
    """
    Custom dataset that loads images (and optionally, metadata features for multimodal) from dataframe.
    Adjust __getitem__ to add metadata if needed.
    """
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        # self.use_metadata = False

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Path to image
        img_name = os.path.join(self.img_dir, self.df.iloc[idx]['image_path'])
        image = Image.open(img_name).convert('RGB')  # Ensure 3 channels

        # Image transforms (incl. normalization & augmentation if train)
        if self.transform:
            image = self.transform(image)

        # Load label as integer (change column if needed)
        label = int(self.df.iloc[idx]['label_int'])

        # Example for adding metadata (uncomment if using multimodal)
        # if self.use_metadata:
        #     meta_features = torch.tensor(self.df.iloc[idx][['age', 'sex_encoded', ...]].values, dtype=torch.float32)
        #     return image, meta_features, label
        # else:
        return image, label

In [ ]:
# labels_df is your full dataframe, 'label_int' is the encoded label column

print("Class counts (full dataset):")
print(labels_df['label_int'].value_counts())

print("\nClass proportions (full dataset):")
print(labels_df['label_int'].value_counts(normalize=True))


Class counts (full dataset):
label_int
5    6705
4    1113
2    1099
1     514
0     327
6     142
3     115
Name: count, dtype: int64

Class proportions (full dataset):
label_int
5    0.669496
4    0.111133
2    0.109735
1    0.051323
0    0.032651
6    0.014179
3    0.011483
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

# Use 100% of the dataset
subset_df = labels_df  # No subsetting

print("Full dataset size:", len(labels_df))
print("Subset (100%) size:", len(subset_df))
print("Class distribution in subset:")
print(subset_df['label_int'].value_counts(normalize=True))


Full dataset size: 10015
Subset (100%) size: 10015
Class distribution in subset:
label_int
5    0.669496
4    0.111133
2    0.109735
1    0.051323
0    0.032651
6    0.014179
3    0.011483
Name: proportion, dtype: float64


In [ ]:
# 2) Now split this subset into train/test
train_df, test_df = train_test_split(
    subset_df,
    test_size=0.15,
    stratify=subset_df['label_int'],
    random_state=42
)

# 3) Split train further into train/val if you currently do that
train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['label_int'],
    random_state=42
)

In [ ]:
# Full dataset size
full_size = len(labels_df)

# Sizes after subsetting and splitting
subset_size = len(subset_df)
train_size  = len(train_df)
val_size    = len(val_df)
test_size   = len(test_df)

print("Full dataset size:", full_size)
print("Train size:", train_size)
print("Val size:", val_size)
print("Test size:", test_size)


# Check that train+val+test equals full
split_total = train_size + val_size + test_size
print("Train+Val+Test total:", split_total)



Full dataset size: 10015
Train size: 7235
Val size: 1277
Test size: 1503
Train+Val+Test total: 10015
